## 16. تشخیص Duplicate

آگهی‌های تکراری می‌توانند حجم عرضه را بیش از واقع نشان دهند.

تیم باید حداقل دو سطح Duplicate را بررسی کند:

1. **Exact Duplicate:** رکوردهای کاملاً یکسان
2. **Probable Duplicate:** آگهی‌های احتمالاً مربوط به یک ملک

ویژگی‌های احتمالی برای Duplicate تقریبی:

- شهر و محله
- مختصات نزدیک
- مساحت
- تعداد اتاق
- طبقه
- قیمت مشابه
- متن مشابه
- ماه ثبت
- نوع کاربر

حذف Duplicate احتمالی باید محافظه‌کارانه و قابل Audit باشد. در صورت عدم حذف، اثر آن بر
شاخص عرضه باید در تحلیل حساسیت بررسی شود.

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist
from itertools import combinations
import hashlib

from datasets import load_dataset

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [3]:
df = pd.read_feather("../Outputs/02_df.feather")

In [24]:
has_location = (
    df["location_latitude"].notna() &
    df["location_longitude"].notna()
)

In [9]:
df["is_exact_duplicate"] = df.duplicated(keep="first")

print("Exact duplicates:", df["is_exact_duplicate"].sum())

Exact duplicates: 11


In [20]:
df["lat_key"] = np.nan
df["lon_key"] = np.nan

df["lat_key"] = df["location_latitude"].round(4)
df["lon_key"] = df["location_longitude"].round(4)

In [21]:
duplicate_cols = [
    "city_slug",
    "neighborhood_slug",
    "cat3_slug",
    "property_type",
    "price_regime",
    "lat_key",
    "lon_key",
    "building_size",
    "land_size",
    "rooms_count",
    "floor"
]



In [29]:
df["duplicate_group"] = np.nan

df.loc[has_location, "duplicate_group"] = (
    df.loc[has_location]
      .groupby(duplicate_cols, dropna=False)
      .ngroup()
)

group_size = (
    df.groupby("duplicate_group")["duplicate_group"]
      .transform("size")
)

df["is_probable_duplicate"] = (
    has_location &
    (group_size > 1)
)

print(
    "Probable duplicate rows:",
    df["is_probable_duplicate"].sum()
)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_19664\681488259.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(duplicate_cols, dropna=False)


Probable duplicate rows: 11320


In [30]:
# ============================================
# 11. Duplicate statistics
# ============================================

total_rows = len(df)

duplicate_rows = df["is_probable_duplicate"].sum()

non_duplicate_rows = total_rows - duplicate_rows

print("=" * 50)
print("Duplicate Detection Results")
print("=" * 50)

print("Total rows:", total_rows)
print("Rows with valid location:", has_location.sum())
print("Rows without location:", (~has_location).sum())
print("Probable duplicate rows:", duplicate_rows)
print("Non-duplicate rows:", non_duplicate_rows)

Duplicate Detection Results
Total rows: 999953
Rows with valid location: 655586
Rows without location: 344367
Probable duplicate rows: 11320
Non-duplicate rows: 988633


In [31]:
# ============================================
# 13. View duplicate records
# ============================================

duplicates = (
    df[df["is_probable_duplicate"]]
    .sort_values("duplicate_group")
)

duplicate_view = duplicates[
    [
        "duplicate_group",
        "city_slug",
        "neighborhood_slug",
        "cat3_slug",
        "property_type",
        "price_regime",
        "building_size",
        "land_size",
        "rooms_count",
        "floor",
        "location_latitude",
        "location_longitude",
        "title"
    ]
]

duplicate_view.head(50)

,duplicate_group,city_slug,neighborhood_slug,cat3_slug,property_type,price_regime,building_size,land_size,rooms_count,floor,location_latitude,location_longitude,title
753476,187.0,Iranshahr,NaN,house-villa-sell,NaN,sell,50.0,300,2,<NA>,27.206242,60.517040,فروش خانه ریکپوت
7860,187.0,Iranshahr,NaN,house-villa-sell,NaN,sell,50.0,300,2,<NA>,27.206242,60.517040,فروش خانه ریکپوت
595678,235.0,Iranshahr,NaN,industry-agriculture-business-sell,NaN,sell,30000.0,<NA>,1,<NA>,27.142574,60.549389,زمین کشاورزی 24 ساعت آب
456638,235.0,Iranshahr,NaN,industry-agriculture-business-sell,NaN,sell,30000.0,<NA>,1,<NA>,27.142574,60.549389,زمین کشاورزی 24 ساعت آب
952375,408.0,Iranshahr,NaN,plot-old,NaN,sell,200.0,<NA>,<NA>,<NA>,27.209621,60.649841,زمین،سجادشهر،200متر
996489,408.0,Iranshahr,NaN,plot-old,NaN,sell,200.0,<NA>,<NA>,<NA>,27.209621,60.649841,زمین،سجادشهر،200متر
939073,441.0,Iranshahr,NaN,plot-old,NaN,sell,250.0,<NA>,<NA>,<NA>,27.215057,60.685749,زمین فروشی استاد یادگاری 18
20979,441.0,Iranshahr,NaN,plot-old,NaN,sell,250.0,<NA>,<NA>,<NA>,27.215057,60.685749,زمین
966709,441.0,Iranshahr,NaN,plot-old,NaN,sell,250.0,<NA>,<NA>,<NA>,27.215057,60.685749,زمین مسکونی
40771,442.0,Iranshahr,NaN,plot-old,NaN,sell,300.0,<NA>,<NA>,<NA>,27.215057,60.685749,زمین 300 متر کلینک اعظم خانی محله سربازی


In [ ]:
# محل پیاده‌سازی تیم:
# 1. exact_duplicate_flag
# 2. probable_duplicate_cluster_id
# 3. deduplication_confidence
# 4. deduplication_rule_version
#
# تعداد آگهی قبل و بعد از Deduplication را گزارش کنید.
#  I want you to compare location_lat location_long + room_count + floor + land size / building size

In [32]:
# ============================================
# 16. Keep one record from each duplicate group
# ============================================

duplicate_mask = df["is_probable_duplicate"]

df_duplicate = df[duplicate_mask].copy()

df_non_duplicate = df[~duplicate_mask].copy()

df_duplicate_keep_one = (
    df_duplicate
    .drop_duplicates(
        subset=duplicate_cols,
        keep="first"
    )
)

df_clean = pd.concat(
    [
        df_non_duplicate,
        df_duplicate_keep_one
    ],
    ignore_index=True
)

print("Original rows:", len(df))
print("Clean rows:", len(df_clean))
print("Removed rows:", len(df) - len(df_clean))

Original rows: 999953
Clean rows: 993927
Removed rows: 6026


In [33]:
# ============================================
# 17. Remove temporary columns
# ============================================

temporary_columns = [
    "is_exact_duplicate",
    "lat_key",
    "lon_key",
    "duplicate_group",
    "is_probable_duplicate"
]

df_clean = df_clean.drop(
    columns=temporary_columns,
    errors="ignore"
)

print("Final shape:", df_clean.shape)

Final shape: (993927, 65)


In [34]:
df_clean.to_feather(
    "../Outputs/03_df.feather"
)